# Coordenadas paralelas das fronteiras verdadeiras

Este notebook reúne os nove cenários sintéticos do benchmark em uma matriz 3×3. As linhas representam pontos da fronteira de Pareto verdadeira. Para permitir a leitura conjunta de objetivos com escalas distintas, cada resposta é normalizada entre o valor ideal verdadeiro (0, melhor valor) e o valor nadir verdadeiro (1, pior valor). A cor de cada linha é determinada exclusivamente por $f_1$ normalizada, usando a mesma escala em todos os painéis.

A subamostragem é determinística e estratificada em $f_1$: ela reduz apenas a densidade gráfica, preserva os extremos e não modifica os arquivos de referência nem os cálculos de desempenho do estudo.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.colors import Normalize

def project_root(start=Path.cwd()):
    path = start.resolve()
    for candidate in (path, *path.parents):
        if (candidate / 'configs' / 'full.json').exists():
            return candidate
    raise FileNotFoundError('Raiz do projeto não encontrada.')

ROOT = project_root()
REFERENCE_DIR = ROOT / 'data' / 'reference_fronts'
OUT_DIR = ROOT / 'results' / 'synthetic' / 'figures' / 'true_pareto'
OUT_DIR.mkdir(parents=True, exist_ok=True)

M_VALUES = (4, 6, 12)
CORRELATIONS = (
    ('low', 'baixa'),
    ('medium', 'média'),
    ('high', 'alta'),
)
DISPLAY_LINES = 900
N_STRATA = 30
PLOT_SEED = 20260825
CMAP = 'RdBu_r'
OUTPUT_STEM = 'nove_cenarios_coordenadas_paralelas_f1_normalizada'
FIGURE_WIDTH_CM = 16.0
FIGURE_HEIGHT_CM = 18.5
CM_TO_INCH = 1 / 2.54

plt.rcParams.update({
    'font.family': 'DejaVu Serif',
    'font.size': 9.5,
    'axes.titlesize': 10.5,
    'axes.labelsize': 10,
    'xtick.labelsize': 8.5,
    'ytick.labelsize': 8.5,
    'savefig.facecolor': 'white',
    'axes.facecolor': 'white',
})

print('Raiz:', ROOT)
print('Saída:', OUT_DIR)

In [ ]:
def scenario_seed(scenario):
    return PLOT_SEED + sum((index + 1) * ord(char) for index, char in enumerate(scenario))

def load_normalized_reference(scenario, expected_m):
    path = REFERENCE_DIR / f'{scenario}_pareto_reference.npz'
    if not path.exists():
        raise FileNotFoundError(f'Referência ausente: {path}')
    with np.load(path, allow_pickle=False) as data:
        F = np.asarray(data['F'], dtype=float)
        ideal = np.asarray(data['ideal_true'], dtype=float)
        nadir = np.asarray(data['nadir_true'], dtype=float)
    assert F.ndim == 2 and F.shape[1] == expected_m, (scenario, F.shape)
    assert ideal.shape == nadir.shape == (expected_m,)
    assert len(F) == 100_000 and np.isfinite(F).all()
    span = nadir - ideal
    if np.any(span <= 0):
        raise ValueError(f'{scenario}: intervalo de normalização inválido.')
    normalized = np.clip((F - ideal) / span, 0.0, 1.0)
    return normalized, ideal, nadir, path

def stratified_display_sample(Fn, scenario, n_lines=DISPLAY_LINES, n_strata=N_STRATA):
    if len(Fn) <= n_lines:
        return Fn.copy()
    rng = np.random.default_rng(scenario_seed(scenario))
    f1 = Fn[:, 0]
    edges = np.linspace(0.0, 1.0, n_strata + 1)
    strata = np.clip(np.digitize(f1, edges[1:-1], right=False), 0, n_strata - 1)
    quota = n_lines // n_strata
    chosen = []
    for stratum in range(n_strata):
        candidates = np.flatnonzero(strata == stratum)
        if len(candidates):
            take = min(quota, len(candidates))
            chosen.extend(rng.choice(candidates, size=take, replace=False).tolist())
    chosen = set(chosen)
    chosen.update((int(np.argmin(f1)), int(np.argmax(f1))))
    remaining = n_lines - len(chosen)
    if remaining > 0:
        pool = np.setdiff1d(np.arange(len(Fn)), np.fromiter(chosen, dtype=int), assume_unique=False)
        chosen.update(rng.choice(pool, size=remaining, replace=False).tolist())
    indices = np.array(sorted(chosen), dtype=int)
    if len(indices) > n_lines:
        protected = {int(np.argmin(f1)), int(np.argmax(f1))}
        removable = np.array(sorted(set(indices) - protected), dtype=int)
        keep = rng.choice(removable, size=n_lines - len(protected), replace=False)
        indices = np.array(sorted(protected | set(keep.tolist())), dtype=int)
    assert len(indices) == n_lines
    return Fn[indices]

def parallel_segments(values):
    x = np.arange(values.shape[1], dtype=float)
    return np.stack((np.broadcast_to(x, values.shape), values), axis=2)

def style_panel(ax, m, scenario):
    x = np.arange(m)
    ax.vlines(x, 0, 1, color='0.55', linewidth=0.75, alpha=0.65, zorder=0)
    ax.set_xlim(-0.15, m - 0.85)
    ax.set_ylim(-0.025, 1.025)
    ax.set_xticks(x, [rf'$f_{{{i}}}$' for i in range(1, m + 1)])
    if m == 12:
        ax.tick_params(axis='x', labelsize=7.2, pad=2)
    ax.set_yticks(np.linspace(0, 1, 6))
    ax.grid(axis='y', color='0.88', linewidth=0.55, zorder=0)
    ax.set_title(scenario, pad=5)
    for spine in ax.spines.values():
        spine.set_color('0.35')
        spine.set_linewidth(0.7)

In [ ]:
fig, axes = plt.subplots(
    3, 3,
    figsize=(FIGURE_WIDTH_CM * CM_TO_INCH, FIGURE_HEIGHT_CM * CM_TO_INCH),
    sharey=True,
    layout='constrained',
)
color_norm = Normalize(vmin=0.0, vmax=1.0)
manifest_rows = []

for row, m in enumerate(M_VALUES):
    for column, (correlation_code, correlation_label) in enumerate(CORRELATIONS):
        scenario = f'm{m}_{correlation_code}'
        Fn, ideal, nadir, source_path = load_normalized_reference(scenario, m)
        shown = stratified_display_sample(Fn, scenario)
        collection = LineCollection(
            parallel_segments(shown),
            cmap=CMAP,
            norm=color_norm,
            linewidths=0.48,
            alpha=0.28,
            rasterized=True,
            zorder=2,
        )
        collection.set_array(shown[:, 0])
        axes[row, column].add_collection(collection)
        style_panel(axes[row, column], m, scenario)
        manifest_rows.append({
            'scenario': scenario,
            'm': m,
            'correlation_level': correlation_code,
            'reference_points': len(Fn),
            'display_lines': len(shown),
            'color_variable': 'f1_normalized',
            'normalization': '(f_j - ideal_true_j) / (nadir_true_j - ideal_true_j)',
            'source': source_path.relative_to(ROOT).as_posix(),
        })

fig.supylabel('Resposta normalizada (0 = ótimo; 1 = pior)', fontsize=10.5)
scalar_mappable = plt.cm.ScalarMappable(norm=color_norm, cmap=CMAP)
scalar_mappable.set_array([])
colorbar = fig.colorbar(
    scalar_mappable, ax=axes, location='bottom', orientation='horizontal',
    shrink=1.0, pad=0.035, aspect=55
)
colorbar.set_label(r'$f_1$ normalizada (0 = ótimo; 1 = pior)', labelpad=12)
colorbar.set_ticks(np.linspace(0, 1, 6))

png_path = OUT_DIR / f'{OUTPUT_STEM}.png'
pdf_path = OUT_DIR / f'{OUTPUT_STEM}.pdf'
fig.savefig(png_path, dpi=300)
fig.savefig(pdf_path, dpi=300)
plt.close(fig)

manifest = pd.DataFrame(manifest_rows)
manifest_path = OUT_DIR / f'{OUTPUT_STEM}_manifest.csv'
manifest.to_csv(manifest_path, index=False)
metadata = {
    'layout': '3x3; rows m=4,6,12; columns low,medium,high',
    'reference': 'true Pareto fronts with 100000 points per scenario',
    'axis_normalization': '(f_j - ideal_true_j) / (nadir_true_j - ideal_true_j), clipped to [0,1]',
    'color': 'f1 normalized with RdBu_r; blue at 0 (optimal) and red at 1 (worst)',
    'publication_size_cm': [FIGURE_WIDTH_CM, FIGURE_HEIGHT_CM],
    'display_sampling': f'deterministic, stratified by normalized f1, {DISPLAY_LINES} lines per panel',
    'seed': PLOT_SEED,
    'png': png_path.relative_to(ROOT).as_posix(),
    'pdf': pdf_path.relative_to(ROOT).as_posix(),
}
metadata_path = OUT_DIR / f'{OUTPUT_STEM}_metadata.json'
metadata_path.write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding='utf-8')

for artifact in (png_path, pdf_path, manifest_path, metadata_path):
    assert artifact.exists() and artifact.stat().st_size > 0
assert len(manifest) == 9 and set(manifest['reference_points']) == {100_000}
print(manifest.to_string(index=False))
print('Arquivos gerados:')
for artifact in (png_path, pdf_path, manifest_path, metadata_path):
    print(' -', artifact.relative_to(ROOT).as_posix())

## Leitura da figura

- Cada linha colorida representa uma solução da referência de Pareto verdadeira.
- Todos os eixos usam a mesma interpretação: 0 corresponde ao ideal verdadeiro do objetivo e 1 ao nadir verdadeiro.
- A barra de cores representa apenas $f_1$ normalizada. Portanto, linhas com a mesma cor possuem desempenho semelhante em $f_1$, embora possam apresentar compromissos distintos nos demais objetivos.
- As linhas da matriz variam o número de objetivos; as colunas organizam os níveis de correlação baixa, média e alta dos nove cenários originais.